# Nano-Tech Artist 단계 2 — Sana-0.6B LoRA 파인튜닝 (Colab)

사전학습된 Sana-0.6B를 우리가 만든 도형 합성 이미지+캡션 데이터로 LoRA 파인튜닝합니다.

**중요**: diffusers의 공식 `train_dreambooth_lora_sana.py`를 그대로 사용합니다 — 학습 루프를 직접 재구현하지 않고 검증된 구현에 우리 데이터만 연결하는 방식입니다 (`docs/15-stage2-pretrained-finetuning.md` 참조). 이 노트북은 huggingface.co 접근이 필요해서 로컬 개발 샌드박스에서는 실행 검증이 불가능했습니다 — 데이터 준비 단계까지는 이미 그 환경에서 실제로 검증되었습니다.

## 1. 리포지토리 클론 (우리 프로젝트 + diffusers 소스)

In [ ]:
!git clone https://github.com/choichoi3227-crypto/cloud-press.git
!git clone https://github.com/huggingface/diffusers.git
%cd cloud-press/ai-models/training/nano-tech-artist

## 2. GPU 확인

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (경고: 매우 느립니다)")

## 3. 라이브러리 설치

In [ ]:
%cd /content/diffusers
!pip install -e .
%cd examples/dreambooth
!pip install -r requirements_sana.txt
!pip install -q bitsandbytes datasets
%cd /content/cloud-press/ai-models/training/nano-tech-artist

In [ ]:
# train_dreambooth_lora_sana.py는 diffusers 0.37.0.dev0 이상을 요구한다 (check_min_version).
# pip install -e .로 소스 설치했으므로 최신 dev 버전이 설치되지만, 확인차 버전을 출력한다.
import diffusers
print("diffusers version:", diffusers.__version__)

import peft
print("peft version:", peft.__version__)
assert tuple(map(int, peft.__version__.split(".")[:2])) >= (0, 14), \
    "peft >= 0.14.0 필요 (requirements_sana.txt 명시 사항)"

## 4. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/cloud-press/checkpoints/nano-tech-artist-lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. 도형 합성 데이터셋 생성 및 캡션 변환
실제 이미지 수집 없이 코드로 도형을 그려서 데이터를 만듭니다 (`docs/04-nano-tech-artist-spec.md` 참조). Sana는 512px 해상도 기준이므로 이미지 크기도 맞춥니다.

In [ ]:
SHAPES_DIR = "/content/drive/MyDrive/cloud-press/data/shapes_512"
CAPTION_DIR = "/content/drive/MyDrive/cloud-press/data/sana_finetune"
!python generate_dataset.py --out {SHAPES_DIR} --count 3000 --size 512
!python prepare_sana_captions.py --shapes-dir {SHAPES_DIR} --out {CAPTION_DIR}

## 6. LoRA 파인튜닝 실행
`run_sana_lora_finetune.py`가 diffusers 공식 스크립트를 우리 데이터로 호출합니다. 체크포인트가 있으면 자동으로 이어서 학습합니다 (`--resume_from_checkpoint=latest`).

In [ ]:
!python run_sana_lora_finetune.py \
    --dataset-dir {CAPTION_DIR} \
    --output-dir {OUTPUT_DIR} \
    --base-model Efficient-Large-Model/Sana_600M_512px_diffusers \
    --max-train-steps 1000

## 7. 생성 결과 확인 (정성 평가)

In [ ]:
import torch
from diffusers import SanaPipeline

pipe = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_600M_512px_diffusers", torch_dtype=torch.bfloat16
)
pipe.load_lora_weights(OUTPUT_DIR)
pipe.to("cuda")

import matplotlib.pyplot as plt
prompts = [
    "a red circle on a white background",
    "a blue triangle on a white background",
    "a simple green square, minimalist icon style",
]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, prompt in zip(axes, prompts):
    image = pipe(prompt=prompt, num_inference_steps=20).images[0]
    ax.imshow(image)
    ax.set_title(prompt, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. CPU 추론 속도 실측 (중요 — docs/15 2.2절의 미확정 항목을 채우는 실측)
공식 자료는 GPU 기준(16GB 노트북 GPU에서 1초 이내)만 존재합니다. 실제 서비스는 CPU로 돌아가야 하므로, 여기서 직접 측정한 수치가 이 프로젝트의 실제 판단 근거가 됩니다.

In [ ]:
import time

cpu_pipe = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_600M_512px_diffusers", torch_dtype=torch.float32
)
cpu_pipe.load_lora_weights(OUTPUT_DIR)
cpu_pipe.to("cpu")

start = time.time()
image = cpu_pipe(prompt="a purple triangle on a white background", num_inference_steps=20).images[0]
elapsed = time.time() - start

print(f"CPU 추론 시간 (512px, 20 steps): {elapsed:.1f}초")
print("이 수치를 docs/15-stage2-pretrained-finetuning.md 2.2절에 실측치로 반영하세요.")
image

## 9. Hugging Face Hub에 LoRA 어댑터 업로드

In [ ]:
from huggingface_hub import login, HfApi

login()

HF_REPO_ID = "<your-username>/nano-tech-artist-lora"  # 실제 사용자명으로 변경

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO_ID)
print("업로드 완료:", HF_REPO_ID)